In [6]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [7]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [8]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optimal : Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
  # Suggest values for the hyperparameters
  n_estimators = trial.suggest_int('n_estimators', 50,200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # Create the RandomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=42
  )

  # Perform 3-fold cross-validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
  
  return score  # Return the accuracy score for Optuma to maximize


In [10]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # we aim to maximize accuracy
study.optimize(objective, n_trials=50) # Run 50 trials to find the best hyperameters

[I 2026-03-22 17:59:18,441] A new study created in memory with name: no-name-620dab0b-6aeb-465c-b8a5-dd7b42f370bd
[I 2026-03-22 17:59:19,161] Trial 0 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 200, 'max_depth': 13}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-03-22 17:59:19,452] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 66, 'max_depth': 16}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-03-22 17:59:19,819] Trial 2 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 114, 'max_depth': 5}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-03-22 17:59:20,403] Trial 3 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 167, 'max_depth': 17}. Best is trial 3 with value: 0.7746741154562384.
[I 2026-03-22 17:59:21,027] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 174, 'max_depth': 11}. Best is trial 3 with value: 0.774674

In [11]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 119, 'max_depth': 18}
